# Solution 5D: CAPM Alpha and Information Ratio
**BUSI 722: Data-Driven Finance II**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

preds = pd.read_parquet("expanding_predictions.parquet")
preds["u"] = preds.groupby("month")["pred"].transform(lambda x: x.rank(pct=True))

## Setup: Compute portfolio returns

In [2]:
# D10-D1 sort-based
preds["decile"] = preds.groupby("month")["pred"].transform(
    lambda x: pd.qcut(x, 10, labels=False, duplicates="drop") + 1)
d10 = preds[preds["decile"] == preds["decile"].max()].groupby("month")["return"].mean()
d1 = preds[preds["decile"] == preds["decile"].min()].groupby("month")["return"].mean()
common = sorted(set(d10.index) & set(d1.index))
ls_sort = d10.loc[common] - d1.loc[common]

# Smooth weights (linear, power, exponential)
smooth = {}
for m, grp in preds.groupby("month"):
    u, ret = grp["u"].values, grp["return"].values
    for name, wf in [("Linear", lambda u: u - 0.5),
                     ("Power", lambda u: u**3 - (u**3).mean()),
                     ("Exponential", lambda u: np.exp(2*u) - np.exp(2*u).mean())]:
        w = wf(u)
        w = w / np.abs(w).sum()
        smooth.setdefault(name, {})[m] = np.sum(w * ret)

    # Score-tilted
    mcap = grp["marketcap"].values
    w = mcap * (u ** 2)
    w = w / w.sum()
    smooth.setdefault("Score-Tilted", {})[m] = np.sum(w * ret)

smooth_df = pd.DataFrame(smooth).sort_index()

## Download Fama-French factors

In [3]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath(".")))
try:
    from ff_utils import download_ff_factors
    ff = download_ff_factors(start_year=2024)
    has_ff = True
    print(f"FF factors: {len(ff)} months")
except Exception as e:
    print(f"Could not load FF factors: {e}")
    has_ff = False

FF factors: 26 months


## 1-3. CAPM regression and Information Ratio

In [4]:
if has_ff:
    portfolios = {
        "Linear (5B)": smooth_df["Linear"],
        "Power (5B)": smooth_df["Power"],
        "Exponential (5B)": smooth_df["Exponential"],
        "D10-D1 Sort (5A)": ls_sort,
        "Score-Tilted (5C)": smooth_df["Score-Tilted"],
    }

    results = []
    for name, port in portfolios.items():
        cm = sorted(set(port.index) & set(ff.index))
        if len(cm) < 3:
            continue
        excess = port.loc[cm].values - ff.loc[cm, "RF"].values
        X = sm.add_constant(ff.loc[cm, "Mkt-RF"].values)
        capm = sm.OLS(excess, X).fit()
        ir = capm.params[0] / capm.resid.std() * np.sqrt(12) if capm.resid.std() > 0 else 0
        results.append({"Portfolio": name, "CAPM Alpha": capm.params[0],
                        "t-stat": capm.tvalues[0], "Beta": capm.params[1],
                        "R2": capm.rsquared, "Ann. IR": ir})
        print(f"{name}: alpha={capm.params[0]:.4f} (t={capm.tvalues[0]:.2f}), "
              f"beta={capm.params[1]:.3f}, IR={ir:.3f}")

    print()
    print(pd.DataFrame(results).to_string(index=False, float_format="%.4f"))
else:
    print("FF factors not available. Install pandas-datareader or provide ff_utils.py.")

Linear (5B): alpha=0.0078 (t=2.25), beta=-0.083, IR=1.846
Power (5B): alpha=0.0043 (t=1.44), beta=-0.066, IR=1.182
Exponential (5B): alpha=0.0058 (t=1.83), beta=-0.075, IR=1.497
D10-D1 Sort (5A): alpha=0.0462 (t=3.16), beta=-0.025, IR=2.592
Score-Tilted (5C): alpha=-0.0024 (t=-2.23), beta=0.952, IR=-1.827

        Portfolio  CAPM Alpha  t-stat    Beta     R2  Ann. IR
      Linear (5B)      0.0078  2.2520 -0.0835 0.0329   1.8464
       Power (5B)      0.0043  1.4418 -0.0655 0.0274   1.1821
 Exponential (5B)      0.0058  1.8256 -0.0750 0.0311   1.4967
 D10-D1 Sort (5A)      0.0462  3.1615 -0.0251 0.0002   2.5920
Score-Tilted (5C)     -0.0024 -2.2290  0.9517 0.9783  -1.8275


## 4. Discussion

The portfolio with the highest annualized information ratio offers the best risk-adjusted alpha. Smooth weight functions (particularly power and exponential) tend to outperform simple sorts because they use the full cross-section of predicted scores rather than discarding information through discrete grouping.